# Weight sensitivity — Prong B (w mixture perturbation)

Perturbs expert **w** (OSM / CORINE / population / … weights inside each group’s **S**), renormalises **W^(g)**, then fuses with **fixed α**.

Requires `build_and_export` bundles that include `mix` term rasters in `groups_manifest.yaml` (re-export after code update).

Loads from `Output/Proxy_diagnostics/W_groups/`.

In [1]:
from pathlib import Path
import importlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import yaml

REPO = Path.cwd()
if not (REPO / "proxy").is_dir():
    REPO = Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import proxy.diagnostics.weight_sensitivity.load_exports as load_exports
importlib.reload(load_exports)
from proxy.diagnostics.weight_sensitivity.load_exports import (
    bundle_path,
    load_bundle,
    mass_for_pollutant,
)
from proxy.diagnostics.weight_sensitivity.pollutants_config import reference_pollutants_from_cfg
import proxy.diagnostics.weight_sensitivity.report_figures as report_figures
importlib.reload(report_figures)
from proxy.diagnostics.weight_sensitivity.report_figures import (
    alpha_row_for_pollutant,
    collect_b4_rows,
    load_prong_a_w_summaries,
    pick_b3_case,
    sectors_with_mix,
    sensitive_for_pollutant,
    headline_w_vs_alpha_stats,
    state_for_pollutant,
)
from proxy.diagnostics.weight_sensitivity.report_figures import make_report_figures
from proxy.diagnostics.weight_sensitivity.plots import (
    apply_thesis_style,
    plot_b3,
    plot_headline_w_vs_alpha_sector,
)


from proxy.diagnostics.weight_sensitivity.prong_b_core import pick_pollutant_index, pick_reference_pollutant


def load_sector_bundle(sector_key: str):
    data = load_bundle(bundle_path(EXPORT_ROOT, sector_key, COUNTRY_TAG, YEAR))
    if not data.get("has_mix"):
        raise ValueError(f"{sector_key}: bundle has no mix terms — re-run build_and_export")
    return sector_key, data


def load_rep_bundle(role: str):
    rep = CFG["representative_sectors"][role]
    sk, data = load_sector_bundle(rep["sector_key"])
    pix = pick_pollutant_index(data["alpha"], rep["primary_pollutant"])
    return sk, data, pix


def primary_pollutant_for_sector(alpha_doc, sector_key: str) -> str:
    ref = reference_pollutants_from_cfg(CFG)
    rep = next(
        (r for r in CFG["representative_sectors"].values() if r["sector_key"] == sector_key),
        None,
    )
    mode = rep["primary_pollutant"] if rep else ref[0]
    return pick_reference_pollutant(alpha_doc, mode, ref)


import proxy.diagnostics.weight_sensitivity.prong_w_core as prong_w_core
importlib.reload(prong_w_core)
from proxy.diagnostics.weight_sensitivity.prong_w_core import (
    collect_w_perturb_tvs,
    collect_w_perturb_tvs_multi,
    prepare_w2_state,
    run_w2_all_representatives,
    run_w2_on_state,
    perturb_weight,
    stack_from_mix,
)
from proxy.core import log
log.configure("INFO")

CFG_PATH = REPO / "proxy" / "config" / "weight_sensitivity_prong_b.yaml"
with CFG_PATH.open(encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

def _export_root(repo: Path) -> Path:
    for name in ("Output", "OUTPUT"):
        p = repo / name / "Proxy_diagnostics" / "W_groups"
        if p.is_dir():
            return p
    return repo / "OUTPUT" / "Proxy_diagnostics" / "W_groups"

EXPORT_ROOT = _export_root(REPO)
COUNTRY_TAG = "Greece"
YEAR = 2019
FIG_DIR = REPO / "Output" / "Proxy_diagnostics" / "figures" / "prong_b_w"
FIG_DIR.mkdir(parents=True, exist_ok=True)

REPORT_POL = CFG["report_pollutant"]

In [6]:
# A1/A2 cross-sector figures (from prong_a_w summary.json — not built by B2/B3/B4 cells)
import proxy.diagnostics.weight_sensitivity.plots as plots_mod
importlib.reload(plots_mod)
from proxy.diagnostics.weight_sensitivity.plots import apply_thesis_style, plot_a1_cross_sector, plot_a2_overlay

apply_thesis_style()
pollutants = reference_pollutants_from_cfg(CFG)
sectors_a = sectors_with_mix(EXPORT_ROOT, COUNTRY_TAG, YEAR)
prong_a_w_a = load_prong_a_w_summaries(REPO, sectors_a, COUNTRY_TAG, YEAR)
if not prong_a_w_a:
    raise RuntimeError("no prong_a_w summaries — run entry.py with RUN_MODE=prong_a_w first")

for pol in pollutants:
    sector_a1 = {}
    a2_slice = []
    for r in prong_a_w_a:
        bp = (r.get("by_pollutant") or {}).get(pol)
        if not bp:
            continue
        sector_a1[r["sector_key"]] = {
            int(k): float(v) for k, v in bp["a1_mass_share_by_active_terms"].items()
        }
        a2_slice.append({
            "sector_key": r["sector_key"],
            "a2_similarity_samples": bp.get("a2_similarity_samples") or [],
            "a2_mass_weights": bp.get("a2_mass_weights") or [],
        })
    if sector_a1:
        plot_a1_cross_sector(sector_a1, FIG_DIR / f"a1_cross_sector__{pol}.png", pollutant=pol)
    if a2_slice:
        plot_a2_overlay(
            a2_slice,
            FIG_DIR / f"a2_overlay__{pol}.png",
            threshold=float(CFG["similarity_threshold"]),
            pollutant=pol,
        )

log.info(f"A1/A2 saved under {FIG_DIR}")
sorted(FIG_DIR.glob("a1_cross_sector__*.png")), sorted(FIG_DIR.glob("a2_overlay__*.png"))

A1/A2 saved under c:\Users\leopi\PDM_local\Output\Proxy_diagnostics\figures\prong_b_w


([WindowsPath('c:/Users/leopi/PDM_local/Output/Proxy_diagnostics/figures/prong_b_w/a1_cross_sector__NMVOC.png'),
  WindowsPath('c:/Users/leopi/PDM_local/Output/Proxy_diagnostics/figures/prong_b_w/a1_cross_sector__NOx.png'),
  WindowsPath('c:/Users/leopi/PDM_local/Output/Proxy_diagnostics/figures/prong_b_w/a1_cross_sector__PM10.png'),
  WindowsPath('c:/Users/leopi/PDM_local/Output/Proxy_diagnostics/figures/prong_b_w/a1_cross_sector__SOx.png')],
 [WindowsPath('c:/Users/leopi/PDM_local/Output/Proxy_diagnostics/figures/prong_b_w/a2_overlay__NMVOC.png'),
  WindowsPath('c:/Users/leopi/PDM_local/Output/Proxy_diagnostics/figures/prong_b_w/a2_overlay__NOx.png'),
  WindowsPath('c:/Users/leopi/PDM_local/Output/Proxy_diagnostics/figures/prong_b_w/a2_overlay__PM10.png'),
  WindowsPath('c:/Users/leopi/PDM_local/Output/Proxy_diagnostics/figures/prong_b_w/a2_overlay__SOx.png')])

In [2]:
from matplotlib.patches import Patch

PCT = 0.2
PAIR_W = 0.22
POL_COLORS = {"PM10": "#4f7cff", "NOx": "#3ecf8e", "SOx": "#f5a623", "NMVOC": "#e05c5c"}

pollutants = reference_pollutants_from_cfg(CFG)
sectors = sectors_with_mix(EXPORT_ROOT, COUNTRY_TAG, YEAR)
prong_a_w = load_prong_a_w_summaries(REPO, sectors, COUNTRY_TAG, YEAR)
log.info(f"B2 violins: {len(sectors)} sectors, {len(pollutants)} pollutants each")

for sk in sectors:
    data = load_bundle(bundle_path(EXPORT_ROOT, sk, COUNTRY_TAG, YEAR))
    state = prepare_w2_state(data, repo_root=REPO)
    n_g = len(state["group_names"])
    by_pol: dict = {}
    for pol in pollutants:
        mass = mass_for_pollutant(data["mass_by_pollutant"], pol)
        cids = [c for c in mass if mass[c] > 0]
        if not cids:
            continue
        sens = sensitive_for_pollutant(prong_a_w, sk, pol) or set()
        by_pol[pol] = {
            "alpha_row": alpha_row_for_pollutant(state["alpha_doc"], pol, n_g),
            "cids": cids,
            "sensitive": sens,
        }
    if not by_pol:
        log.warning(f"B2 skip {sk}: no CAMS mass")
        continue

    tv_by_pol = collect_w_perturb_tvs_multi(state, by_pol, PCT, label=sk)
    active = [p for p in pollutants if p in tv_by_pol and tv_by_pol[p][0]]
    if not active:
        log.warning(f"B2 skip {sk}: no TV samples")
        continue

    positions: list[float] = []
    datasets: list[list[float]] = []
    colors: list[str] = []
    for i, pol in enumerate(active):
        tv_all, tv_sens = tv_by_pol[pol]
        c = POL_COLORS.get(pol, "#8b91a8")
        center = float(i + 1)
        positions.append(center - PAIR_W / 2)
        datasets.append(tv_all)
        colors.append(c)
        if tv_sens:
            positions.append(center + PAIR_W / 2)
            datasets.append(tv_sens)
            colors.append(c)

    fig, ax = plt.subplots(figsize=(2.4 * len(active) + 2.5, 4.8))
    parts = ax.violinplot(datasets, positions=positions, widths=PAIR_W, showmeans=True, showextrema=False)
    for body, col in zip(parts["bodies"], colors):
        body.set_facecolor(col)
        body.set_edgecolor(col)
        body.set_alpha(0.72)

    ax.set_xticks(range(1, len(active) + 1))
    ax.set_xticklabels(active)
    ax.set_ylabel("TV(c)")
    ax.set_title(
        f"B2 local +/-{int(PCT * 100)}% w — {sk}\n"
        "left violin = all cells | right = mix-sensitive",
        fontsize=10,
    )
    ax.legend(
        handles=[Patch(facecolor=POL_COLORS[p], edgecolor=POL_COLORS[p], label=p) for p in active],
        loc="upper right",
        fontsize=8,
        frameon=False,
    )
    fig.tight_layout()
    out = FIG_DIR / f"b2_violin__{sk}.png"
    fig.savefig(out, dpi=150)
    plt.close(fig)
    log.info(f"B2 saved {out.name}")



B2 violins: 5 sectors, 4 pollutants each
Prong B (w): loaded 556 mix-sensitive cells from prong_a_w


KeyboardInterrupt: 

In [3]:
# B2 summary stats (report_pollutant from YAML — uses matching CAMS mass + alpha row)
importlib.reload(prong_w_core)
from proxy.diagnostics.weight_sensitivity.prong_w_core import run_w2_all_representatives

summaries = run_w2_all_representatives(EXPORT_ROOT, REPO, CFG, COUNTRY_TAG, YEAR)
(FIG_DIR / "b2_summary.json").write_text(json.dumps(summaries, indent=2), encoding="utf-8")
log.info(f"b2_summary.json written ({CFG['report_pollutant']} for all roles)")
summaries


Prong B (w) role=public_power sector=A_PublicPower pollutant=NMVOC
Prong B (w): loaded 556 mix-sensitive cells from prong_a_w
Prong B (w) A_PublicPower NMVOC: fuse baseline W0 (1 groups, alpha fixed)
Prong B (w) A_PublicPower NMVOC: perturb 4/4
Prong B (w) role=industry sector=B_Industry pollutant=NMVOC
Prong B (w): loaded 613 mix-sensitive cells from prong_a_w
Prong B (w) B_Industry NMVOC: fuse baseline W0 (4 groups, alpha fixed)
Prong B (w) B_Industry NMVOC: perturb 4/32
Prong B (w) B_Industry NMVOC: perturb 8/32
Prong B (w) B_Industry NMVOC: perturb 12/32
Prong B (w) B_Industry NMVOC: perturb 16/32
Prong B (w) B_Industry NMVOC: perturb 20/32
Prong B (w) B_Industry NMVOC: perturb 24/32
Prong B (w) B_Industry NMVOC: perturb 28/32
Prong B (w) B_Industry NMVOC: perturb 32/32
Prong B (w) role=waste sector=J_Waste pollutant=NMVOC
Prong B (w): loaded 3319 mix-sensitive cells from prong_a_w
Prong B (w) J_Waste NMVOC: fuse baseline W0 (3 groups, alpha fixed)
Prong B (w) J_Waste NMVOC: pertur

{'public_power': {'sector': 'A_PublicPower',
  'pollutant': 'NMVOC',
  'b2': {'analysis': 'w_mixture',
   'tv_all': {'mean': 0.00386678560502438, 'p90': 0.01219081564922818},
   'tv_sensitive': {'mean': 0.0047013414883655446,
    'p90': 0.014368438862584302},
   'n_sensitive_cells': 556,
   'mix_a3_nox_frac': None,
   'pollutant': 'NMVOC',
   'group_names': ['public_power']}},
 'industry': {'sector': 'B_Industry',
  'pollutant': 'NMVOC',
  'b2': {'analysis': 'w_mixture',
   'tv_all': {'mean': 0.006657062840332843, 'p90': 0.023585186511400038},
   'tv_sensitive': {'mean': 0.007341230630963126, 'p90': 0.024857657040001868},
   'n_sensitive_cells': 613,
   'mix_a3_nox_frac': None,
   'pollutant': 'NMVOC',
   'group_names': ['refineries_petroleum',
    'manufacturing_combustion_residual',
    'mineral',
    'chemical_metal']}},
 'waste': {'sector': 'J_Waste',
  'pollutant': 'NMVOC',
  'b2': {'analysis': 'w_mixture',
   'tv_all': {'mean': 0.0008621566578381332, 'p90': 0.0017055180407623998}

In [4]:
# Headline — w vs alpha per sector (report_pollutant; skip A_PublicPower)
PCT_HEADLINE = 0.2
apply_thesis_style()
sectors_h = [s for s in sectors_with_mix(EXPORT_ROOT, COUNTRY_TAG, YEAR) if s != "A_PublicPower"]
prong_a_w_h = load_prong_a_w_summaries(REPO, sectors_h, COUNTRY_TAG, YEAR)
log.info(f"Headline w vs alpha: {len(sectors_h)} sectors, pollutant={REPORT_POL}")

for sk in sectors_h:
    data = load_bundle(bundle_path(EXPORT_ROOT, sk, COUNTRY_TAG, YEAR))
    state = prepare_w2_state(data, repo_root=REPO)
    sp = state_for_pollutant(state, mass_for_pollutant(data["mass_by_pollutant"], REPORT_POL))
    sens = sensitive_for_pollutant(prong_a_w_h, sk, REPORT_POL)
    if sens:
        sp["sensitive"] = sens
    if not sp["cids"]:
        log.warning(f"Headline skip {sk}: no {REPORT_POL} CAMS mass")
        continue
    stats = headline_w_vs_alpha_stats(sp, REPORT_POL, PCT_HEADLINE)
    log.info(
        f"Headline {sk}: tv_w mean={stats['tv_w_mean']:.4g} p90={stats['tv_w_p90']:.4g} "
        f"tv_a mean={stats['tv_a_mean']:.4g} p90={stats['tv_a_p90']:.4g}"
    )
    out = FIG_DIR / f"headline_w_vs_alpha__{sk}__{REPORT_POL}.png"
    plot_headline_w_vs_alpha_sector(sk, stats, out, pollutant=REPORT_POL, pct=PCT_HEADLINE)
    log.info(f"Headline saved {out.name}")



Headline w vs alpha: 4 sectors, pollutant=NMVOC
Prong B (w): loaded 613 mix-sensitive cells from prong_a_w
Headline B_Industry: tv_w mean=0.004366 p90=0.01582 tv_a mean=0.008262 p90=0.02972
Headline saved headline_w_vs_alpha__B_Industry__NMVOC.png
Prong B (w): loaded 760 mix-sensitive cells from prong_a_w
Headline D_Fugitive: tv_w mean=0.0005188 p90=1.228e-08 tv_a mean=0.02395 p90=0.04385
Headline saved headline_w_vs_alpha__D_Fugitive__NMVOC.png
Prong B (w): loaded 2807 mix-sensitive cells from prong_a_w
Headline I_Offroad: tv_w mean=0.0006131 p90=0.001255 tv_a mean=0.01284 p90=0.03018
Headline saved headline_w_vs_alpha__I_Offroad__NMVOC.png
Prong B (w): loaded 3319 mix-sensitive cells from prong_a_w
Headline J_Waste: tv_w mean=0.0005553 p90=0.001166 tv_a mean=0.01341 p90=0.04572
Headline saved headline_w_vs_alpha__J_Waste__NMVOC.png


In [9]:
# B3 — mix-sensitive cell patch (one figure per sector, max-elasticity weight)
B3_PCT = 0.4
sectors_b3 = sectors_with_mix(EXPORT_ROOT, COUNTRY_TAG, YEAR)
prong_a_w_b3 = load_prong_a_w_summaries(REPO, sectors_b3, COUNTRY_TAG, YEAR)
log.info(f"B3 case studies: {len(sectors_b3)} sectors")

for sk in sectors_b3:
    data = load_bundle(bundle_path(EXPORT_ROOT, sk, COUNTRY_TAG, YEAR))
    state = prepare_w2_state(data, repo_root=REPO)
    pol = primary_pollutant_for_sector(state["alpha_doc"], sk)
    n_g = len(state["group_names"])
    sp = state_for_pollutant(state, mass_for_pollutant(data["mass_by_pollutant"], pol))
    sens = sensitive_for_pollutant(prong_a_w_b3, sk, pol)
    if sens:
        sp["sensitive"] = sens
    if not sp["cids"] or not sp["sensitive"]:
        log.warning(f"B3 skip {sk}: no CAMS mass or mix-sensitive cells")
        continue
    alpha_row = alpha_row_for_pollutant(state["alpha_doc"], pol, n_g)
    try:
        p0, p1, shape, _cid, title, _tv = pick_b3_case(sp, alpha_row, sk, pol, pct=B3_PCT)
        out = FIG_DIR / f"b3_case_study__{sk}.png"
        plot_b3(p0, p1, shape, out, title=title)
        plot_b3(p0, p1, shape, out.with_suffix(".pdf"), title=title)
        log.info(f"B3 saved {out.name}")
    except ValueError as exc:
        log.warning(f"B3 skip {sk}: {exc}")


B3 case studies: 5 sectors
Prong B (w): loaded 556 mix-sensitive cells from prong_a_w
B3 saved b3_case_study__A_PublicPower.png
Prong B (w): loaded 613 mix-sensitive cells from prong_a_w
B3 saved b3_case_study__B_Industry.png
Prong B (w): loaded 760 mix-sensitive cells from prong_a_w
B3 saved b3_case_study__D_Fugitive.png
Prong B (w): loaded 321 mix-sensitive cells from prong_a_w
B3 saved b3_case_study__I_Offroad.png
Prong B (w): loaded 3319 mix-sensitive cells from prong_a_w
B3 saved b3_case_study__J_Waste.png


In [5]:
# B4 — elasticity dTV/dw (+/-20%, report_pollutant for all sectors)
import pandas as pd

rows = []
pct = 0.2
sectors_b4 = sectors_with_mix(EXPORT_ROOT, COUNTRY_TAG, YEAR)
for sk in sectors_b4:
    data = load_bundle(bundle_path(EXPORT_ROOT, sk, COUNTRY_TAG, YEAR))
    state = prepare_w2_state(data, repo_root=REPO)
    n_g = len(state["group_names"])
    sp = state_for_pollutant(state, mass_for_pollutant(data["mass_by_pollutant"], REPORT_POL))
    if not sp["cids"]:
        log.warning(f"B4 skip {sk}: no {REPORT_POL} CAMS mass")
        continue
    alpha_row = alpha_row_for_pollutant(state["alpha_doc"], REPORT_POL, n_g)
    for _pol, sector, gname, wkey, dtv in collect_b4_rows(sp, alpha_row, sk, REPORT_POL, pct):
        rows.append({"sector": sector, "pollutant": _pol, "group": gname, "weight": wkey, "dTV_dw": dtv})

df = pd.DataFrame(rows)
display(df)
df.to_csv(FIG_DIR / "b4_elasticity.csv", index=False)
log.info(f"B4 saved b4_elasticity.csv ({len(df)} rows, pollutant={REPORT_POL})")



Prong B (w): loaded 556 mix-sensitive cells from prong_a_w
Prong B (w): loaded 613 mix-sensitive cells from prong_a_w
Prong B (w): loaded 760 mix-sensitive cells from prong_a_w
Prong B (w): loaded 2807 mix-sensitive cells from prong_a_w
Prong B (w): loaded 3319 mix-sensitive cells from prong_a_w


,sector,pollutant,group,weight,dTV_dw
0,A_PublicPower,NMVOC,public_power,w1,9.746433e-03
1,B_Industry,NMVOC,refineries_petroleum,w_osm,2.813621e-02
2,B_Industry,NMVOC,refineries_petroleum,w_pop,3.234371e-02
3,B_Industry,NMVOC,manufacturing_combustion_residual,w_osm,5.406204e-02
4,B_Industry,NMVOC,manufacturing_combustion_residual,w_pop,4.709182e-02
5,B_Industry,NMVOC,mineral,w_osm,5.308860e-05
6,B_Industry,NMVOC,mineral,w_pop,7.267643e-05
7,B_Industry,NMVOC,chemical_metal,w_osm,3.973312e-04
8,B_Industry,NMVOC,chemical_metal,w_pop,5.467324e-04
9,D_Fugitive,NMVOC,coal_and_solid_fuels,w_osm,4.722228e-03


B4 saved b4_elasticity.csv (57 rows, pollutant=NMVOC)
